# Visualize one GaitLU-1M shard

This notebook reads the first silhouette sequence from one raw `.tar.gz` shard and displays several frames. It streams the archive, so it does not unpack the whole shard.

Use this only with trusted GaitLU files: the raw sequences are stored as Python pickle files, and loading a pickle can execute code.

In [ ]:
from pathlib import Path
import pickle
import tarfile

import matplotlib.pyplot as plt
import numpy as np

# Change this to the shard you want to inspect.
shard_path = Path("/hai/scratch/tedmui/cody-jepa/data/gaitlu-1m/shards0/gaitlu-000.tar.gz")

if not shard_path.is_file():
    raise FileNotFoundError(f"Shard does not exist: {shard_path}")

print(f"Reading: {shard_path}")

In [ ]:
def decode_sequence(value):
    # GaitLU pickles may store the frame array under one of these keys.
    if isinstance(value, dict):
        for key in ("silhouettes", "frames", "data"):
            if key in value:
                value = value[key]
                break
        else:
            raise ValueError(f"No frame data found in dictionary: {list(value)}")

    # Handle a one-item wrapper such as [frames].
    if isinstance(value, (list, tuple)) and len(value) == 1:
        candidate = np.asarray(value[0])
        if candidate.ndim >= 3:
            value = value[0]

    frames = np.asarray(value)

    # Convert [T, 1, H, W] or [T, H, W, 1] to [T, H, W].
    if frames.ndim == 4 and frames.shape[1] == 1:
        frames = frames[:, 0]
    elif frames.ndim == 4 and frames.shape[-1] == 1:
        frames = frames[..., 0]

    if frames.ndim != 3 or 0 in frames.shape:
        raise ValueError(f"Expected non-empty [time, height, width], got {frames.shape}")

    # Normalize either 0/1 or 0/255-style silhouettes to 0/1.
    if frames.max() <= 1:
        return frames >= 0.5
    return frames >= 128

sequence_name = None
frames = None

# r|* supports .tar.gz, .tgz, and .tar while streaming member-by-member.
with tarfile.open(shard_path, mode="r|*") as archive:
    for member in archive:
        if not member.isfile() or not member.name.lower().endswith(".pkl"):
            continue

        handle = archive.extractfile(member)
        if handle is None:
            raise OSError(f"Could not read archive member: {member.name}")

        # Only do this for trusted, official GaitLU archives.
        with handle:
            sequence = pickle.load(handle)

        sequence_name = member.name
        frames = decode_sequence(sequence)
        break

if frames is None:
    raise RuntimeError(
        "No .pkl files were found. This may be a prepared bit-packed shard, "
        "which requires its inventory CSV."
    )

print(f"Sequence: {sequence_name}")
print(f"Frames:   {frames.shape[0]}")
print(f"Size:     {frames.shape[1]} x {frames.shape[2]} pixels")

## Contact sheet

This shows evenly spaced frames from the sequence, which is usually the quickest way to check what the data looks like.

In [ ]:
frames_to_show = min(8, len(frames))
indices = np.linspace(0, len(frames) - 1, frames_to_show).astype(int)

fig, axes = plt.subplots(
    1,
    frames_to_show,
    figsize=(2 * frames_to_show, 4),
    squeeze=False,
)

for axis, index in zip(axes[0], indices):
    axis.imshow(
        frames[index],
        cmap="gray",
        vmin=0,
        vmax=1,
        interpolation="nearest",
    )
    axis.set_title(f"Frame {index}")
    axis.axis("off")

fig.suptitle(sequence_name)
plt.tight_layout()
plt.show()

## Optional: play the sequence

Run the next cell if you want to see the person move frame by frame.

In [ ]:
from IPython.display import HTML
from matplotlib import animation

fig, axis = plt.subplots(figsize=(3, 4))
axis.axis("off")
image = axis.imshow(
    frames[0],
    cmap="gray",
    vmin=0,
    vmax=1,
    interpolation="nearest",
)

def update(index):
    image.set_data(frames[index])
    return (image,)

movie = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=100,
    blit=True,
)

plt.close(fig)
HTML(movie.to_jshtml())